# 20 - Government: The Rules Live Outside the Record

Program eligibility is determined by statute — income
thresholds, categorical rules, exceptions, effective dates —
that live in law and policy manuals, not in the data. The
applicant record has the fields; nothing in it encodes which
rule version applies or when it changed.

Here the statutes are versioned, date-effective POLICIES layered
over the record:

| Mechanic | How it is served |
| --- | --- |
| Statute versions | validity windows (`effectiveFrom`/`Until`) |
| "As of the application date" | `data_access_context.as_of` |
| Categorical over income test | priority bands, visible in rank |
| Documented exceptions | the steward request lane |

Program mechanics are illustrative by design.


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
if mode == "live" and not os.getenv("METATATE_MCP_URL"):
    print("Live mode needs a Metatate endpoint. Fastest path (about 5 minutes):")
    print("  1. Create a free account: https://app.getmetatate.com/sign-up?ref=examples")
    print("  2. Workspace dashboard: 'Load the demo' banner -> 'Load the Customer 360 demo'")
    print("  3. MCP Tools -> Tokens: issue a token; Connect tab has your endpoint URL")
    print("  4. export METATATE_MCP_URL=... METATATE_SAAS_MCP_TOKEN=...")
    print("     (full steps: docs/live-mode-saas.md)")

client = get_client()
print(f"Metatate examples mode: {mode}")


PRODUCT_DATABASE_TABLES = {"product_usage_events", "support_tickets", "ml_feature_store"}


def asset(table, column=None, schema="public", database=None):
    resolved_database = database or (
        "product" if table in PRODUCT_DATABASE_TABLES else "master"
    )
    ref = {"database": resolved_database, "schema": schema, "table": table}
    if column:
        ref["column"] = column
    return ref


def answer_label(answer):
    state = answer.get("state")
    if state and state != "answered":
        return state
    return answer.get("decision") or answer.get("verdict") or "unknown"


def print_answer(answer):
    print(f"state:    {answer.get('state')}")
    if "decision" in answer:
        print(f"decision: {answer['decision']}")
    if "verdict" in answer:
        print(f"verdict:  {answer['verdict']}")
    if answer.get("reason"):
        print(f"reason:   {answer['reason']}")
    for condition in answer.get("conditions") or []:
        print(f"condition [{condition.get('kind')}]: {condition.get('requirement')}")
    for prohibition in answer.get("prohibitions") or []:
        print(f"prohibition: {prohibition.get('detail')}")
    for obligation in answer.get("obligations") or []:
        print(f"obligation [{obligation.get('type')}]: {obligation.get('target')}")
    if "can_proceed_now" in answer:
        print(f"can_proceed_now: {answer['can_proceed_now']}")


## 1. Which statute answers today


In [ ]:
today = client.authorize_use(
    asset("applicants", database="benefits"),
    use="determine benefits eligibility for an application",
    scenario_key="compliance.regulatory",
    purpose_key="benefits.determination",
)
print("today ->", answer_label(today))
for row in today.get("instructions", []):
    print(
        f"  {row['provenance']['policy_name']:28}"
        f" effective_until={row.get('effective_until')}"
    )


## 2. The as-of flip: evaluate under the statute in force at T


In [ ]:
as_of_2027 = client.authorize_use(
    asset("applicants", database="benefits"),
    use="determine benefits eligibility for an application",
    scenario_key="compliance.regulatory",
    purpose_key="benefits.determination",
    data_access_context={"as_of": "2027-02-01T00:00:00Z"},
)
condition = next(
    (c for c in as_of_2027.get("conditions", [])
     if c.get("kind") == "ai_restriction"),
    {},
)
print("as of 2027-02-01 ->", answer_label(as_of_2027))
print("  requirement:", condition.get("requirement", ""))
print("  validity_evaluated_at:", as_of_2027.get("validity_evaluated_at"))


Same call, one added instant: the 2025 statute leaves force and
the 2026 statute answers — with the partition instant stated in
the provenance. The record never changed; the RULES did.


## 3. Published but not in force


In [ ]:
not_yet = client.authorize_use(
    asset("qualifying_conditions", database="benefits"),
    use="apply the modernized categorical checklist",
    scenario_key="compliance.regulatory",
    purpose_key="benefits.determination",
)
print("qualifying_conditions ->", not_yet["state"], "/", not_yet["reason_code"])


## 4. Managed precedence: categorical over the income test


In [ ]:
precedence = client.authorize_use(
    asset("applicants", database="benefits"),
    use="run an automated benefits determination",
    scenario_key="ai.automated_decisioning",
    purpose_key="benefits.determination",
)
print("determination ->", answer_label(precedence))
for row in precedence.get("instructions", []):
    print(
        f"  priority={row['priority']} "
        f"{row['provenance']['policy_name']:32} {row['decision']}"
    )


The categorical permit (critical band) outranks the income test
(high band) in the ranked instructions — precedence is
governed, versioned, and visible. And the composition is
fail-safe: the income-test CONDITION still surfaces on the
answer, so a permitted determination never silently skips it.


## 5. Managed precedence vs unmanaged disagreement


In [ ]:
conflict = client.authorize_use(
    asset("applicants", database="benefits"),
    use="run verification outreach against applicants",
    scenario_key="purpose.allowed_use",
)
print("verification outreach ->", conflict["state"], "/", conflict["reason_code"])
sources = (conflict.get("conflict") or {}).get("sources", [])
for source in sources:
    print("  conflicting:", source["provenance"]["policy_name"])


Contrast: two same-band policies disagree about verification
outreach, and the estate surfaces the CONFLICT with both
sources cited instead of silently picking a winner. Managed
precedence is governance; unmanaged disagreement is debt made
visible. Documented exceptions follow the steward request lane
(notebook 09): an exception can satisfy a review or a
condition — it never overrides a deny.


## 6. The as-of instant flips the SQL verdict too


In [ ]:
DETERMINATION_SQL = "SELECT applicant_id, categorical_flag FROM benefits.public.applicants"

sql_today = client.validate_query_context(
    DETERMINATION_SQL,
    scenario_key="compliance.regulatory",
    default_database="master", default_schema="public",
    purpose_key="benefits.determination",
)
sql_2027 = client.validate_query_context(
    DETERMINATION_SQL,
    scenario_key="compliance.regulatory",
    default_database="master", default_schema="public",
    purpose_key="benefits.determination",
    data_access_context={"as_of": "2027-02-01T00:00:00Z"},
)
print("today          ->", sql_today.get("verdict"), "/", sql_today.get("state"))
print("as of 2027     ->", sql_2027.get("verdict"), "/", sql_2027.get("state"))


## 7. The receipt


In [ ]:
receipt = client.explain_why(
    authorization_id=as_of_2027["authorization_id"],
)
print("decision  :", receipt["decision"], "/", receipt["answer_state"])
print("cited rows:", len(receipt["cited_decision_ids"]))
print("evaluated :", receipt["provenance"]["evaluated_at"])


"Which rule version applied?" is answerable after the fact: the
durable record carries the policy version, its effective
window, and the instant the validity partition used.
